# Event Response Analysis: Prediction Markets vs. Polls

**DS2500 — Programming with Data | Northeastern University**  
**Team:** Sebastian Brookes, Shiven Mishra

How did prediction markets (Polymarket) and traditional polls (FiveThirtyEight) respond to
major campaign events during the 2024 U.S. presidential election? This notebook measures
**reaction speed**, **intensity**, and **directional accuracy** for five key events.


In [ ]:
# ---------------------------------------------------------------------------
# Import analysis helpers and configure plotting for the notebook
# ---------------------------------------------------------------------------

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# Add the project root to sys.path so notebook cells can import from src/.
PROJECT_ROOT = str(Path.cwd().parent)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.clean.utils import PROCESSED_DIR, SWING_STATES
from src.analysis.events.event_response import (
    EVENTS,
    OVERLAP_CUTOFF,
    get_latest_state_snapshot,
    load_daily_data,
    compute_swing_average,
    build_fivethirtyeight_swing_series,
    compute_event_reaction,
    compute_raw_indexed_window,
    build_reaction_summary,
)
from src.visualize.event_plots import (
    configure_plot_style,
    plot_event_timeline,
    plot_reaction_scoreboard,
    plot_indexed_event_study,
)

configure_plot_style()

plt.rcParams["figure.dpi"] = 150

## Methodology

### Events analyzed

We focus on **five major campaign events** that fell within the **head-to-head window**
(through Sept 12, 2024), when both Polymarket and FiveThirtyEight have daily data for
swing states:

| #   | Event                       | Date    | Expected Direction |
| --- | --------------------------- | ------- | ------------------ |
| 1   | Biden-Trump Debate          | June 27 | pro-Trump          |
| 2   | Trump Assassination Attempt | July 13 | pro-Trump          |
| 3   | Biden Drops Out             | July 21 | pro-Harris         |
| 4   | Walz VP Pick                | Aug 6   | neutral            |
| 5   | Harris-Trump Debate         | Sept 10 | pro-Harris         |

### Why swing states?

National polls and probabilities wash out the signal: California will go blue and Alabama
will go red regardless of campaign events. By averaging across the seven swing states
(AZ, GA, MI, NV, NC, PA, WI), we are able to inspect the _marginal_ effect of each event on the
states that actually decide the election.

### Measuring reactions

For each event, we compute:

- **Baseline** — average Trump lead over the 3 days before the event
- **Immediate** — average Trump lead on the event day and the day after
- **Settled** — average Trump lead over days 2–7 after the event
- **Shift** — settled minus baseline (the sustained change)
- **z-score** — shift divided by the source's daily volatility (standard deviation of
  day-over-day changes), so we can compare across sources on a common scale

### How the pipeline works

The functions imported above (from `src/analysis/events/event_response.py`) walk through
the main data-wrangling steps before we ever make a plot.

1. **Loading comparable state-level data.** `load_daily_data()` reads the processed daily
   Polymarket and FiveThirtyEight files. It also removes FiveThirtyEight's national
   `US` row so the analysis stays focused on states rather than mixing state and national
   numbers together.

2. **Turning each source into one swing-state time series.** `compute_swing_average()`
   averages Polymarket's `trump_lead` across the seven swing states for each day.
   `build_fivethirtyeight_swing_series()` does the same for FiveThirtyEight, but first it has to
   **forward-fill** missing dates: for each state and each day, it uses the most recent poll
   value available up to that day. That gives us a daily 538 series we can compare to the
   daily Polymarket series through the shared overlap cutoff of September 12.

3. **Measuring each event with fixed windows.** `compute_event_reaction()` looks at one
   event date at a time and computes a baseline (days -3 to -1), an immediate reaction
   (days 0 to +1), and a settled reaction (days +2 to +7). The main quantity we care
   about is the **shift**, which is settled minus baseline. A positive shift means the
   source moved toward Trump; a negative shift means it moved toward Harris.

4. **Checking data quality before scoring the event.** An event only counts if the source
   has pre-event data and full coverage in the post-event window (`has_data`). This rule is
   why FiveThirtyEight cannot fully score the September 10 Harris-Trump debate: the overlap
   window ends on September 12, so most of the days needed for the settled window are missing.

5. **Turning shifts into judgments.** `classify_direction()` compares the observed shift to
   the event's expected direction. For pro-Trump events, a positive shift is correct; for
   pro-Harris events, a negative shift is correct. For neutral events, a large move counts as
   an overreaction.

6. **Building the summary table.** `build_reaction_summary()` repeats this process for all
   five events and both sources, then computes `shift_z`. Here, **volatility** means a
   source's normal day-to-day wiggle, and a **z-score** tells us how large the event reaction
   was relative to that usual noise.

7. **Indexing the dropout response chart.** For the final event-study plot, the notebook
   recenters each source so Day -1 equals zero. That lets us compare response timing after
   Biden dropped out without getting distracted by the fact that Polymarket and
   FiveThirtyEight use different scales.

In [ ]:
# ---------------------------------------------------------------------------
# Walkthrough: tracing one event from processed daily rows to the final score
# ---------------------------------------------------------------------------

event_name = "Biden Drops Out"
event_date = "2024-07-21"
state = "PA"
as_of = "2024-07-24"

pm_walk, p538_walk = load_daily_data(PROCESSED_DIR)
pm_swing = compute_swing_average(pm_walk, SWING_STATES)
p538_swing = build_fivethirtyeight_swing_series(p538_walk, SWING_STATES, OVERLAP_CUTOFF)

window_start = pd.Timestamp(event_date) - pd.Timedelta(days=3)
window_end = pd.Timestamp(event_date) + pd.Timedelta(days=7)

# --- Step 1: Inspect the processed state-level rows around the event ---
# Polymarket publishes one row per state per day, so the PA slice is a
# complete daily sequence. FiveThirtyEight only adds a new row when a new
# poll average is published, so its processed daily table can have gaps.
pm_state = pm_walk[(pm_walk["state"] == state) &
                   (pm_walk["date"].between(window_start, window_end))]
p538_state = p538_walk[(p538_walk["state"] == state) &
                       (p538_walk["date"].between(window_start, window_end))]

print(f"── Step 1: Processed daily rows for {state} around {event_name} ──")
print("Polymarket rows:")
display(pm_state[["date", "state", "trump_prob", "trump_lead"]])
print("FiveThirtyEight rows:")
display(p538_state[["date", "state", "trump_pct", "dem_pct", "trump_lead"]])

# --- Step 2: Show how forward-fill creates a daily 538 series ---
# On July 24 there is no new PA polling row, so an exact filter returns 0.
# get_latest_state_snapshot finds the most recent row on or before that date
# and uses it as the value for the reconstructed daily time series.
p538_exact = p538_walk[(p538_walk["state"] == state) &
                      (p538_walk["date"] == pd.Timestamp(as_of))]
p538_filled = get_latest_state_snapshot(p538_walk, as_of)
p538_filled = p538_filled[p538_filled["state"] == state]

print(f"── Step 2: Forward-fill 538 for {state} on {as_of} ──")
print(f"   Exact 538 row on {as_of}: {len(p538_exact)}")
print(f"   Forward-filled row used in the daily series: {len(p538_filled)} "
      f"(from {p538_filled['date'].iloc[0].date()})")
display(p538_filled[["date", "state", "trump_pct", "dem_pct", "trump_lead"]])

swing_window = pd.DataFrame({
    "Polymarket swing avg": pm_swing.loc[window_start:window_end],
    "538 swing avg": p538_swing.loc[window_start:window_end],
})
print("\nThese daily swing-state averages are the inputs to the event-reaction function:")
display(swing_window)

# --- Step 3: Compute the event windows and the sustained shift ---
# compute_event_reaction applies the baseline / immediate / settled windows
# to each swing-state series and returns the shift plus a has_data flag.
pm_reaction = compute_event_reaction(pm_swing, event_date)
p538_reaction = compute_event_reaction(p538_swing, event_date)

print(f"── Step 3: Reaction metrics for {event_name} ──")
for source, reaction in [("Polymarket", pm_reaction),
                         ("FiveThirtyEight", p538_reaction)]:
    print(
        f"   {source:<16} baseline = {reaction['baseline']:.4f}, "
        f"immediate = {reaction['immediate']:.4f}, "
        f"settled = {reaction['settled']:.4f}, "
        f"shift = {reaction['shift']:+.4f}, "
        f"has_data = {reaction['has_data']}"
    )

# --- Step 4: Connect those numbers to the notebook outputs ---
# build_reaction_summary repeats Step 3 for every event and both sources,
# then adds the z-score and direction label used in the scoreboard. The
# indexed window powers the final dropout chart by forcing Day -1 to zero.
summary = build_reaction_summary(EVENTS, pm_walk, p538_walk, SWING_STATES, OVERLAP_CUTOFF)
dropout_rows = summary[summary["event"] == event_name][
    ["source", "baseline", "settled", "shift", "shift_z",
     "direction_match", "has_data"]
]

indexed_window = pd.DataFrame({
    "Polymarket (percentage points)": compute_raw_indexed_window(
        pm_swing, event_date, pre_days=1, post_days=5, scale=100
    ),
    "538 (vote-share points)": compute_raw_indexed_window(
        p538_swing, event_date, pre_days=1, post_days=5, scale=1
    ),
})

print(f"── Step 4: Final outputs for {event_name} ──")
print("Summary-table rows:")
display(dropout_rows)
print("Indexed values for the dropout chart (Day -1 is forced to zero):")
display(indexed_window)

In [ ]:
# ---------------------------------------------------------------------------
# Load the event-response datasets and preview their structure
# ---------------------------------------------------------------------------

pm, p538 = load_daily_data(PROCESSED_DIR)

print(f"Polymarket:      {pm.shape[0]:,} rows, {pm.shape[1]} cols")
print(f"FiveThirtyEight: {p538.shape[0]:,} rows, {p538.shape[1]} cols")
print()
print("Polymarket sample:")
display(pm.head())
print("\nFiveThirtyEight sample:")
display(p538.head())

In [ ]:
# ---------------------------------------------------------------------------
# Aggregate both sources into comparable swing-state time series
# ---------------------------------------------------------------------------

pm_swing = compute_swing_average(pm, SWING_STATES)
p538_swing = build_fivethirtyeight_swing_series(p538, SWING_STATES, OVERLAP_CUTOFF)

print(f"Polymarket swing avg:      {pm_swing.index.min().date()} to {pm_swing.index.max().date()}  ({len(pm_swing)} days)")
print(f"FiveThirtyEight swing avg: {p538_swing.index.min().date()} to {p538_swing.index.max().date()}  ({len(p538_swing)} days)")

In [ ]:
# ---------------------------------------------------------------------------
# Plot the swing-state timeline with key campaign events annotated
# ---------------------------------------------------------------------------

plot_event_timeline(pm_swing, p538_swing, show=True)


### Timeline interpretation

Both sources follow the same broad narrative arc: Trump builds a lead through June and
early July (consistent with Biden's poor debate performance and sympathy after the
assassination attempt), then **Biden dropping out on July 21 appears to be the single
biggest inflection point in the data**. Both series reverse sharply as Harris replaces
Biden.

Key observations:

- **Polymarket appears to have reacted almost immediately** to the dropout, consistent with
  traders repricing overnight. The FiveThirtyEight average took several days to reflect the
  shift, as new polls trickled in.
- Both sources show a modest Harris bump from the September debate, though the signal is much
  weaker than the dropout effect.
- The dual-axis alignment keeps the zero line shared by both series so that "above zero =
  Trump leads" and "below zero = Harris leads" is visually consistent.


In [ ]:
# ---------------------------------------------------------------------------
# Build and format event-level reaction summary table
# ---------------------------------------------------------------------------

summary = build_reaction_summary(EVENTS, pm, p538, SWING_STATES, OVERLAP_CUTOFF)

display_cols = ["event", "source", "expected", "baseline", "settled",
                "shift", "shift_z", "direction_match", "has_data"]
# Format baseline and settled levels, sustained shifts, and z-scores for readability.
display(summary[display_cols].style.format({
    "baseline": "{:.4f}",
    "settled": "{:.4f}",
    "shift": "{:+.4f}",
    "shift_z": "{:+.1f}σ",
}).set_caption("Reaction summary: all events × both sources"))

In [ ]:
# ---------------------------------------------------------------------------
# Visualize reaction intensity and directional accuracy by event and source
# ---------------------------------------------------------------------------

plot_reaction_scoreboard(summary, show=True)


### Scoreboard interpretation

The scoreboard reveals two important patterns:

1. **Direction accuracy** — Polymarket shifted in the expected direction for 4 of 5 measurable events, while
   FiveThirtyEight managed 3 of 4 measurable events. The Harris-Trump debate is excluded from
   FiveThirtyEight's score because the post-event window is incomplete. Both sources showed an unexpected pro-Trump reaction to
   the Walz VP pick (labelled neutral/expected, so any large move counts as overreaction).

2. **Intensity** — Polymarket's z-scored reactions are consistently larger, suggesting that
   markets moved further relative to their own normal daily noise. This is consistent with
   markets processing new information more quickly than poll aggregates, which smooth over
   days.

3. **Data gaps** — FiveThirtyEight has a coverage gap around the Harris-Trump debate
   (Sept 10), which falls near the end of our overlap window (Sept 12). This limits our
   ability to measure the settled reaction for that event in the polling data.


In [ ]:
# ---------------------------------------------------------------------------
# Compare the indexed post-event trajectories for both swing-state series
# ---------------------------------------------------------------------------

plot_indexed_event_study(pm_swing, p538_swing, show=True)


### Biden dropout interpretation

The indexed event study makes the speed difference vivid:

- **Polymarket** began repricing on Day 0 (July 21), but the steepest single-day drops
  came on **Days 1 and 3**, not Day 0 itself. Day 0 was a Sunday afternoon announcement
  with relatively low trading volume. Day 1 (Monday) brought a full trading session plus
  Harris's rapid consolidation of delegate endorsements, giving traders concrete
  information to act on. The Day 3 drop coincides with the wave of major endorsements
  (Pelosi, Schumer, and the Congressional Black Caucus all backed Harris within 72
  hours), which appears to have reduced the remaining uncertainty about a contested
  convention.
- **FiveThirtyEight** took several more days to reflect the shift. The lag annotation on
  the chart shows the gap between each source's steepest single-day drop.

**Why the difference?** Polymarket trades 24/7, so traders can reprice as soon as news
breaks. But even markets don't fully reprice on a single day: the dropout itself was only
the first domino, and each subsequent endorsement cascade narrowed the remaining
uncertainty. FiveThirtyEight aggregates polls, and polls take time to field, collect
responses, weight, and publish. This structural latency suggests that poll aggregates
tend to lag the underlying shift by at least a few days in this dataset, and events that
unfold over multiple news cycles appear to compound the delay.


## Conclusions

### Summary of findings

1. **Markets appear to respond faster.** In this dataset, Polymarket repriced swing-state
   probabilities within hours of major events, while FiveThirtyEight's poll aggregate
   appears to have lagged by days.

2. **Markets were more often aligned with our expected event effects.** Polymarket shifted in the expected
   direction for 4 of 5 measurable events; FiveThirtyEight matched expectations for 3 of 4,
   with the September 10 debate excluded because the post-event window is incomplete.

3. **Markets appear to react more sharply.** Polymarket's z-scored response magnitudes are
   consistently larger, suggesting that market prices carry a stronger reaction relative to
   their own daily volatility compared to polls.

### Limitations

- **Small event sample** — only 5 events, which limits statistical power for accuracy.
- **Overlap window ends Sept 12** — we miss the final two months of the campaign where late
  polls converge toward election day.
- **538 data gaps** — irregular poll publication creates coverage gaps, especially near the
  end of our window (Harris-Trump debate).
- **Mismatched scale** — Polymarket reports win probabilities while 538 reports vote
  shares. Z-scoring normalizes this, but the underlying quantities are still fundamentally
  different.

### So what?

The practical takeaway is that, in this dataset, markets and polls appear to answer
different questions on different timescales. A newsroom building a live election dashboard
might consider **weighting market data for breaking-event coverage** (assassination
attempts, candidate withdrawals, debate nights) and **leaning on poll data for weekly or
monthly trend reports** (such as demographic shifts). Neither source alone tells the full
story, but using both without accounting for the apparent speed difference risks producing
misleading analysis.

For campaigns themselves, these results suggest that if a candidate's internal tracking
relies solely on poll aggregates, they may be seeing the world on a multi-day delay during
the moments that matter most. Market prices won't indicate *why* the race is shifting,
but they appear to signal *that* it is shifting, potentially days before polls can confirm
it.
